*© Sabrine Bendimerad · [sbendimerad.github.io](https://sbendimerad.github.io) · Licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): you may share and adapt this material with credit, for non-commercial use, under the same licence. Datasets keep their own licences.*

# Topic modeling on newsgroups 📰

In this exercise, you run LSA on 2,400 messages to extract topics without any label. Then you compare the result with the newsgroup where each message was posted. We never use the newsgroup to train a model: only to check whether the topics found by LSA look like the categories a human would have chosen.

## The dataset 🗂️
`data/newsgroups.csv` contains 2,400 messages of the *20 Newsgroups* dataset (Usenet discussion groups of the early 1990s, collected by Ken Lang), 400 from each of 6 groups: `comp.graphics`, `rec.autos`, `rec.sport.hockey`, `sci.med`, `sci.space` and `talk.politics.mideast`. Columns: `text` (the message, with headers, signatures and quoted replies removed) and `newsgroup`.

## Loading the data
1. Import the libraries.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import spacy
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import Normalizer

import plotly.io as pio
import plotly.graph_objects as go
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

2. Load the dataset. How many messages are there in each newsgroup?

In [ ]:
news = pd.read_csv("data/newsgroups.csv")
news.head()

,text,newsgroup
0,What would all of you out there in net land th...,sci.space
1,I aparantly mistyped the address for the ftp s...,comp.graphics
2,Also remember that every dollar spent keeping ...,sci.space
3,This is ok in my opinion as long as the stuff ...,sci.space
4,I was skimming through a few gophers and bumpe...,comp.graphics


In [ ]:
news["newsgroup"].value_counts()

newsgroup
sci.space                400
comp.graphics            400
rec.sport.hockey         400
sci.med                  400
talk.politics.mideast    400
rec.autos                400
Name: count, dtype: int64

## Preparing the texts
3. Create a column `clean`: the text in lowercase, with only the letters (replace everything else by a space).

💡 Hint: `news["text"].str.lower().str.replace(r"[^a-z]+", " ", regex=True)`.

In [ ]:
news["clean"] = news["text"].str.lower().str.replace(r"[^a-z]+", " ", regex=True).str.strip()
news[["clean", "newsgroup"]].head()

,clean,newsgroup
0,what would all of you out there in net land th...,sci.space
1,i aparantly mistyped the address for the ftp s...,comp.graphics
2,also remember that every dollar spent keeping ...,sci.space
3,this is ok in my opinion as long as the stuff ...,sci.space
4,i was skimming through a few gophers and bumpe...,comp.graphics


4. Load spaCy's `en_core_web_sm` model, without the components you don't need (`disable=["parser", "ner"]`). Create a column `lemmas` that contains, for each message, the lemmas of its tokens that are not stop words and have at least 3 letters, joined by spaces.

💡 Hint: `nlp.pipe(news["clean"], batch_size=200)` processes the texts in batches, much faster than calling `nlp` on each text. A token has the attributes `.lemma_` and `.is_stop`, and `len(token)` gives its number of characters.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
news["lemmas"] = [" ".join(token.lemma_ for token in doc if not token.is_stop and len(token) > 2)
                  for doc in nlp.pipe(news["clean"], batch_size=200)]
news[["lemmas", "newsgroup"]].head()

,lemmas,newsgroup
0,net land think big martin mariatta boeing mcdo...,sci.space
1,aparantly mistype address ftp site hold image ...,comp.graphics
2,remember dollar spend keep spacecraft safe mod...,sci.space
3,opinion long stuff return earth turn true time...,sci.space
4,skim gopher bump nih database include image gi...,comp.graphics


5. Build the TF-IDF matrix `X` with `TfidfVectorizer(min_df=5, max_df=0.5)`. What is its shape, and what share of its values are zeros?

In [ ]:
vectorizer = TfidfVectorizer(min_df=5, max_df=0.5)
X = vectorizer.fit_transform(news["lemmas"])
words = vectorizer.get_feature_names_out()
print("Shape:", X.shape)
print(f"Share of zeros: {1 - X.nnz / (X.shape[0] * X.shape[1]):.2%}")

Shape: (2400, 3573)
Share of zeros: 98.99%


## LSA
6. Run a `TruncatedSVD` with 10 topics (`random_state=0`) and store the coordinates of the messages in `doc_topics`. What share of the variance do the 10 topics keep? What do you notice?

In [ ]:
svd = TruncatedSVD(n_components=10, random_state=0)
doc_topics = svd.fit_transform(X)
print("Explained variance ratio of each topic:", svd.explained_variance_ratio_.round(4))
print(f"Total: {svd.explained_variance_ratio_.sum():.1%}")

Explained variance ratio of each topic: [0.0024 0.0081 0.0066 0.0061 0.0047 0.0039 0.0038 0.0034 0.0031 0.003 ]
Total: 4.5%


**Each topic keeps less than 1% of the variance, and the 10 topics only 4.5% in total. Text is very high-dimensional: each message has its own words. That's normal, and not a problem: we want a few themes we can read, not a perfect copy of the matrix. Notice that topic 1 keeps the least variance (0.24%): it captures the average message, not the differences between messages.**

7. For each topic, display the 8 words with the largest weights and the 5 words with the smallest (most negative) weights. Give a name to the topics you can.

💡 Hint: `svd.components_[j]` holds the weights of the words in topic `j`; `np.argsort(weights)[::-1][:8]` gives the positions of the 8 largest.

In [ ]:
for j, weights in enumerate(svd.components_):
    print(f"topic {j + 1:2d} (+) {', '.join(words[np.argsort(weights)[::-1][:8]])}")
    print(f"         (-) {', '.join(words[np.argsort(weights)[:5]])}")

topic  1 (+) know, like, think, game, don, car, good, year
         (-) subsystem, characteristic, counterpart, israelites, deputy
topic  2 (+) game, team, play, hockey, espn, player, fan, win
         (-) file, car, image, program, graphic
topic  3 (+) israel, arab, jews, israeli, jewish, people, kill, state
         (-) file, image, format, program, graphic
topic  4 (+) car, engine, dealer, buy, new, oil, drive, price
         (-) file, israel, arab, image, format
topic  5 (+) car, file, israel, arab, jews, format, game, israeli
         (-) space, cause, effect, food, doctor
topic  6 (+) team, player, play, space, good, nhl, league, orbit
         (-) game, espn, doctor, cause, effect
topic  7 (+) space, game, espn, orbit, shuttle, launch, moon, mission
         (-) player, team, good, msg, play
topic  8 (+) file, format, image, problem, tiff, convert, gif, cview
         (-) graphic, thank, card, look, information
topic  9 (+) israel, image, arab, color, cause, problem, line, bit
 

**Topic 1: common words of every group (*know, like, think, game, don*): the "average message", not a theme. Topic 2: hockey (+) against graphics and cars (−). Topic 3: Middle East politics (+) against graphics (−). Topic 4: cars. Topic 7: space (with two hockey words, *game* and *espn*). Topic 8: graphic file formats (*file, format, image, tiff, gif*). Topic 10: graphics cards (*card, color, bit, driver, mode*), with *muslim* mixed in. Topics 5, 6 and 9 are mixed (topic 6 puts *team, player, nhl* together with *space, orbit*). No topic is about medicine.**

## Is each message about its strongest topic?
8. Assign each message to the topic where its coordinate is the largest (`argmax`), in a column `topic`. How many messages does each topic get? What do you notice?

In [ ]:
news["topic"] = doc_topics.argmax(axis=1) + 1        # + 1 to number the topics from 1
news["topic"].value_counts().sort_index()

topic
1     1648
2      178
3      143
4      137
5       11
6       51
7       89
8       48
9       38
10      57
Name: count, dtype: int64

**Topic 1 gets 1,648 of the 2,400 messages (69%), and topic 5 only 11. The "strongest" topic is nearly always the average-message topic.**

9. Cross the newsgroups with the topics (`pd.crosstab`) and draw the table as a heatmap. Compute its **purity**: for each topic, count the messages of its most common newsgroup, add these counts up and divide by the number of messages (1 = each topic contains a single newsgroup). Do the topics match the newsgroups?

💡 Hint: `table = pd.crosstab(news["newsgroup"], news["topic"])`, `px.imshow(table, text_auto=True)`, and the purity is `table.max().sum() / table.values.sum()`.

In [ ]:
table = pd.crosstab(news["newsgroup"], news["topic"])
fig = px.imshow(table, text_auto=True, color_continuous_scale="Blues", aspect="auto")
fig.update_layout(title="Newsgroup vs strongest LSA topic", xaxis_title="strongest topic", height=420)
fig.show()
print(f"Purity: {table.max().sum() / table.values.sum():.3f}")

Purity: 0.452


**The purity is only 0.452: read this way, the topics look very different from the newsgroups. Topic 1 swallows 379 of the 400 sci.med messages, 304 space, 292 graphics, 260 cars, 242 politics and 171 hockey messages. But look at the other columns: they are almost pure. Topic 2 contains 178 hockey messages and nothing else, topic 3 142 politics messages (+1), topic 4 136 car messages (+1), topic 6 51 hockey messages, topic 7 88 space messages (+1). The topics do match newsgroups; it's the argmax that is the wrong way to read them.**

10. Why does topic 1 win so often? Check: what share of the messages have a positive coordinate on topic 1, and what are the average coordinates of each newsgroup on the 10 topics?

In [ ]:
print("Messages with a positive coordinate on topic 1:", (doc_topics[:, 0] > 0).sum(), "of", len(doc_topics))
coords = pd.DataFrame(doc_topics, columns=[f"topic {j + 1}" for j in range(10)])
coords.groupby(news["newsgroup"]).mean().round(2)

Messages with a positive coordinate on topic 1: 2400 of 2400


,topic 1,topic 2,topic 3,topic 4,topic 5,topic 6,topic 7,topic 8,topic 9,topic 10
newsgroup,,,,,,,,,,
comp.graphics,0.13,-0.07,-0.10,-0.06,0.03,0.00,-0.00,-0.02,0.01,0.01
rec.autos,0.14,-0.04,-0.00,0.12,0.04,-0.01,-0.00,0.00,-0.00,0.00
rec.sport.hockey,0.15,0.16,-0.03,-0.01,0.01,0.02,-0.02,-0.00,0.00,-0.01
sci.med,0.13,-0.03,0.01,-0.00,-0.08,-0.05,-0.05,0.01,0.02,-0.01
sci.space,0.12,-0.03,-0.00,0.00,-0.06,0.04,0.08,0.02,-0.00,-0.00
talk.politics.mideast,0.13,-0.03,0.12,-0.06,0.04,0.00,0.00,-0.00,-0.01,0.01


**All 2,400 messages have a positive coordinate on topic 1, and every newsgroup has about the same average on it (0.12 to 0.15). On its own theme topic, a group is about as high or lower: 0.16 for hockey on topic 2, 0.12 for politics on topic 3, 0.12 for cars on topic 4, only 0.08 for space on topic 7. So topic 1 wins the argmax for most messages, without saying anything about their theme. Second problem: argmax only looks at positive values, while a theme can be on the negative side of a topic (graphics is −0.10 on average on topic 3, medicine −0.08 on topic 5) and the signs are arbitrary anyway. LSA coordinates are not "probabilities of belonging to a topic".**

## Clustering in the topic space
A topic is a direction, not a group. To make groups, let's run a clustering on the topic coordinates.

11. Normalize the rows of `doc_topics` with `Normalizer()` (each message becomes a vector of length 1), then run `KMeans` with 6 clusters (`n_init=10`, `random_state=0`). Cross the clusters with the newsgroups, draw the heatmap and compute the purity. Compare with the purity of the same k-means without the `Normalizer`.

💡 Hint: after `Normalizer`, the distance between two messages only depends on the angle between them (their cosine similarity): k-means compares the *mix of topics* of the messages, not how long or how "ordinary" they are.

In [ ]:
doc_topics_norm = Normalizer().fit_transform(doc_topics)
news["cluster"] = KMeans(n_clusters=6, n_init=10, random_state=0).fit_predict(doc_topics_norm)
table = pd.crosstab(news["newsgroup"], news["cluster"])
fig = px.imshow(table, text_auto=True, color_continuous_scale="Blues", aspect="auto")
fig.update_layout(title="Newsgroup vs k-means cluster (normalized LSA coordinates)", xaxis_title="cluster", height=420)
fig.show()
print(f"Purity with the Normalizer: {table.max().sum() / table.values.sum():.3f}")

raw_clusters = KMeans(n_clusters=6, n_init=10, random_state=0).fit_predict(doc_topics)
raw_table = pd.crosstab(news["newsgroup"], raw_clusters)
print(f"Purity without the Normalizer: {raw_table.max().sum() / raw_table.values.sum():.3f}")

Purity with the Normalizer: 0.875
Purity without the Normalizer: 0.610


**With the `Normalizer`, the purity jumps to 0.875: each newsgroup gets its own cluster (hockey 375 of 400 messages in one cluster, politics 370, graphics 349, space 345, medicine 336, cars 324). Without it, the purity is only 0.610: k-means then also groups the messages by the length of their vector (how many typical words they use), not only by their mix of topics.**

12. Which newsgroups does k-means find well? Which ones does it mix up, and why could that be?

**Hockey (375) and politics (370) are found best: they have a very specific vocabulary (team names, *nhl*, *israel*...). Cars (324) and medicine (336) are the weakest. Medicine has no topic of its own, so its cluster also collects the messages with no clear theme: 22 car messages, 19 space, 12 graphics, 9 hockey and 9 politics messages. Short messages, or messages written with ordinary words, are hard to place whatever their newsgroup.**

## Bonus: NMF
13. Run an `NMF` with 6 topics (`init="nndsvda"`, `max_iter=500`, `random_state=0`) on `X`. Display the 8 words with the largest weights of each topic, assign each message to its strongest NMF topic, and compute the purity. Compare with LSA.

In [ ]:
nmf = NMF(n_components=6, init="nndsvda", max_iter=500, random_state=0)
doc_topics_nmf = nmf.fit_transform(X)
for j, weights in enumerate(nmf.components_):
    print(f"NMF topic {j + 1}: {', '.join(words[np.argsort(weights)[::-1][:8]])}")

news["nmf_topic"] = doc_topics_nmf.argmax(axis=1) + 1
table = pd.crosstab(news["newsgroup"], news["nmf_topic"])
fig = px.imshow(table, text_auto=True, color_continuous_scale="Blues", aspect="auto")
fig.update_layout(title="Newsgroup vs strongest NMF topic", xaxis_title="strongest NMF topic", height=420)
fig.show()
print(f"Purity: {table.max().sum() / table.values.sum():.3f}")

NMF topic 1: think, know, like, space, don, people, time, thing
NMF topic 2: game, espn, hockey, fan, baseball, win, watch, night


NMF topic 3: israel, arab, jews, israeli, jewish, kill, state, right
NMF topic 4: car, engine, buy, dealer, new, drive, price, oil
NMF topic 5: file, image, format, program, graphic, color, thank, gif
NMF topic 6: team, player, play, good, nhl, league, year, hockey


Purity: 0.727


**The NMF topics are easy to name, with no negative weights: hockey games (*game, espn, hockey, fan*), Israel, cars, graphics, hockey teams (*team, player, nhl, league*), plus a general topic (*think, know, like, space...*). The argmax works directly and gives a purity of 0.727, much better than the argmax of LSA (0.452) but below k-means on the LSA coordinates (0.875). With only 6 topics, NMF spends two of them on the hockey group (169 and 213 messages) and none on medicine or space: 388 medicine and 363 space messages end up in the general topic.**

## Conclusion
14. What do you conclude about the topics found by LSA and the newsgroups?

**It depends on how you read the topics. Taking each message's strongest LSA topic gives topics that look very different from the newsgroups (purity 0.452), because topic 1 is the average message and wins almost every time. But the topics themselves are meaningful: hockey, Middle East politics, cars, space and graphics each have a clear topic, and a k-means on the normalized topic coordinates recovers the 6 newsgroups with a purity of 0.875, without ever seeing a label. LSA also finds structure that the labels hide (graphic file formats vs graphics cards) and misses groups without a specific vocabulary (medicine has no topic). Remember: LSA topics are directions, a message can mix several of them, and to make groups you run a clustering on top of them.**